# Week 5 Experiments

This notebook runs the prompt-driven baseline required for the Week 5 milestone.

The project uses one Qwen-VL model for both roles:

- text-only prompt: works like the LLM stage
- text + sampled frames prompt: works like the VLM stage


In [ ]:
# Run this once in a fresh environment.
# !pip install -r ../requirements.txt


In [ ]:
from pathlib import Path
import json
import sys
import pandas as pd
import numpy as np

# Let the notebook import from the project root.
PROJECT_ROOT = Path('..').resolve()
sys.path.append(str(PROJECT_ROOT))

from src.run_pipeline import run
from src.rag_annotator import RAGAnnotator
from evaluation.metrics import evaluate_predictions

## 1. Choose a Video

Place a supervisor sample video in `data/videos/`, then update the path below.

In [ ]:
video_path = PROJECT_ROOT / 'data' / 'videos' / 'sample.mp4'
settings_path = PROJECT_ROOT / 'configs' / 'settings.yaml'

video_path


## 2. Run the Week 5 Baseline

This may take time because Qwen-VL loads a large model.

In [ ]:
result = run(str(video_path), str(settings_path))
result['video_id'], len(result['segments'])


In [ ]:
# Show the first segment in a readable way.
if result['segments']:
    print(json.dumps(result['segments'][0], indent=2)[:3000])


## 3. Evaluate Against Ground Truth

Create a ground-truth JSON file with this simple format:

```json
[
  {"start_time": 0.0, "end_time": 2.0, "action_label": "open_hand"}
]
```

In [ ]:
ground_truth_path = PROJECT_ROOT / 'data' / 'annotations' / f"{result['video_id']}_ground_truth.json"

if ground_truth_path.exists():
    ground_truth = json.loads(ground_truth_path.read_text(encoding='utf-8'))
    metrics = evaluate_predictions(result['segments'], ground_truth, iou_threshold=0.5)
else:
    metrics = {
        'status': 'missing_ground_truth',
        'expected_file': str(ground_truth_path),
        'prediction_segments': len(result['segments']),
    }

metrics


In [ ]:
# Save the Week 5 baseline result.
baseline_path = PROJECT_ROOT / 'evaluation' / 'baseline_results.json'
baseline_path.write_text(json.dumps(metrics, indent=2), encoding='utf-8')
baseline_path


## 4. RAG Baseline (Alternative to Prompt-Based)

Now run RAG retrieval on the same video chunks. This uses the action library
directly without any VLM inference, making it faster and fully deterministic.

In [ ]:
# Run RAG annotation on the same chunks
# Note: result['segments'] already have features from the prompt baseline run

rag_annotator = RAGAnnotator()
rag_segments = rag_annotator.annotate_chunks_batch(result['segments'], verbose=True)

print(f"\n✓ RAG annotation complete for {len(rag_segments)} segments")
print(f"  Sample RAG annotation: {rag_segments[0]['rag_annotation']}")


## 5. Compare RAG vs Prompt Baseline

Evaluate both methods on the same ground truth using:
- **Temporal IoU**: How well temporal boundaries align
- **Macro-F1**: Per-action F1 score averaged across all actions


In [ ]:
if ground_truth_path.exists():
    ground_truth = json.loads(ground_truth_path.read_text(encoding='utf-8'))
    
    # Extract prompt baseline annotations (from run result)
    prompt_predictions = result['segments']
    
    # Extract RAG annotations
    rag_predictions = [{
        'start_time': seg['start_time'],
        'end_time': seg['end_time'],
        'action_label': seg['rag_annotation']['action_label'],
    } for seg in rag_segments]
    
    # Evaluate both methods
    prompt_metrics = evaluate_predictions(prompt_predictions, ground_truth, iou_threshold=0.5)
    rag_metrics = evaluate_predictions(rag_predictions, ground_truth, iou_threshold=0.5)
    
    print("=" * 80)
    print("PROMPT BASELINE (Qwen-VL) METRICS")
    print("=" * 80)
    for key, value in prompt_metrics.items():
        if isinstance(value, float):
            print(f"  {key}: {value:.4f}")
        else:
            print(f"  {key}: {value}")
    
    print("\n" + "=" * 80)
    print("RAG BASELINE (Retrieval-Augmented) METRICS")
    print("=" * 80)
    for key, value in rag_metrics.items():
        if isinstance(value, float):
            print(f"  {key}: {value:.4f}")
        else:
            print(f"  {key}: {value}")
    
    # Summary comparison
    print("\n" + "=" * 80)
    print("COMPARISON SUMMARY")
    print("=" * 80)
    comparison = {
        'Metric': ['Mean Temporal IoU', 'Macro-F1'],
        'Prompt': [prompt_metrics['mean_temporal_iou'], prompt_metrics['macro_f1']],
        'RAG': [rag_metrics['mean_temporal_iou'], rag_metrics['macro_f1']],
    }
    df_comparison = pd.DataFrame(comparison)
    df_comparison['Difference (RAG - Prompt)'] = df_comparison['RAG'] - df_comparison['Prompt']
    df_comparison['Percent Diff'] = (df_comparison['Difference (RAG - Prompt)'] / df_comparison['Prompt']) * 100
    print(df_comparison.to_string(index=False))
    
else:
    print("⚠️  Ground truth file not found. Cannot compute metrics.")
    print(f"   Expected: {ground_truth_path}")
    print(f"   Create a JSON file with action segments for metric evaluation.")


## 6. Segment-by-Segment Analysis

In [ ]:
# Create a detailed comparison table for each segment
comparison_data = []
for i, (prompt_seg, rag_seg) in enumerate(zip(prompt_predictions, rag_predictions)):
    comparison_data.append({
        'Segment': i,
        'Start': prompt_seg['start_time'],
        'End': prompt_seg['end_time'],
        'Prompt Label': prompt_seg.get('action_label', 'unknown'),
        'RAG Label': rag_seg['action_label'],
        'Match': prompt_seg.get('action_label', 'unknown') == rag_seg['action_label'],
        'Prompt Conf': prompt_seg.get('confidence', 0.0),
        'RAG Conf': rag_seg.get('confidence', 0.0),
    })

df_segments = pd.DataFrame(comparison_data)
print("\nDetailed Segment Comparison:")
print(df_segments.to_string(index=False))

print(f"\n✓ Label agreement: {df_segments['Match'].sum()}/{len(df_segments)} ({100*df_segments['Match'].sum()/len(df_segments):.1f}%)")


## 7. RAG Retrieval Details

In [ ]:
# Show top alternatives from RAG retrieval for segments where prompt and RAG disagree
print("Where Prompt and RAG Disagree - RAG Top 3 Candidates:")
print("=" * 100)
for i, (prompt_seg, rag_seg) in enumerate(zip(prompt_predictions, rag_predictions)):
    if prompt_seg.get('action_label', 'unknown') != rag_seg['action_label']:
        print(f"\nSegment {i} ({rag_seg['start_time']:.2f}s - {rag_seg['end_time']:.2f}s)")
        print(f"  Prompt said: {prompt_seg.get('action_label', 'unknown')} (conf: {prompt_seg.get('confidence', 0.0):.3f})")
        
        # Get RAG retrieval results
        retrieval = rag_segments[i].get('rag_retrieval', [])
        print(f"  RAG candidates:")
        for rank, candidate in enumerate(retrieval[:3], 1):
            print(f"    {rank}. {candidate['label']} ({candidate['action_id']}) - {candidate['confidence']:.3f}")
            if rank == 1:
                # Show evidence breakdown for top candidate
                print(f"       Evidence: {candidate['evidence']}")


## 8. Save Comparison Results

In [ ]:
# Save comparison results
comparison_results = {
    'video_id': result['video_id'],
    'prompt_baseline': prompt_metrics,
    'rag_baseline': rag_metrics,
    'segment_comparison': comparison_data,
    'summary': {
        'temporal_iou_improvement': rag_metrics['mean_temporal_iou'] - prompt_metrics['mean_temporal_iou'],
        'macro_f1_improvement': rag_metrics['macro_f1'] - prompt_metrics['macro_f1'],
        'label_agreement_rate': float(df_segments['Match'].sum() / len(df_segments)),
    }
}

results_path = PROJECT_ROOT / 'evaluation' / 'rag_vs_prompt_comparison.json'
results_path.parent.mkdir(parents=True, exist_ok=True)
results_path.write_text(json.dumps(comparison_results, indent=2), encoding='utf-8')

print(f"✓ Saved comparison results to: {results_path}")
print(f"\nKey metrics:")
print(f"  Temporal IoU (Prompt): {prompt_metrics['mean_temporal_iou']:.4f}")
print(f"  Temporal IoU (RAG):    {rag_metrics['mean_temporal_iou']:.4f}")
print(f"  Macro-F1 (Prompt):     {prompt_metrics['macro_f1']:.4f}")
print(f"  Macro-F1 (RAG):        {rag_metrics['macro_f1']:.4f}")
